In [1]:
# ==== Cell 1: Setup (RUN THIS FIRST) ====
!pip -q install datasets transformers sentencepiece nltk dateparser icalendar rouge-score spacy==3.7.4
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""   # force CPU for stability

from transformers import pipeline

# Small, fast models
SUM_MODEL = "sshleifer/distilbart-cnn-12-6"          # summarization
ZSC_MODEL = "typeform/distilbert-base-uncased-mnli"  # zero-shot classification

# Pipelines
summarizer = pipeline("summarization", model=SUM_MODEL, device=-1)
print(f"[OK] summarizer loaded: {SUM_MODEL}")

zsc = pipeline("zero-shot-classification", model=ZSC_MODEL, device=-1)
print(f"[OK] zero-shot classifier loaded: {ZSC_MODEL}")

# NLP helpers
import nltk, spacy, dateparser, re, json, datetime
nltk.download("punkt", quiet=True)
try: nltk.download("punkt_tab", quiet=True)
except Exception: pass
nlp = spacy.load("en_core_web_sm")
print("[OK] spaCy model loaded")

# Output dir
from pathlib import Path
OUT_DIR = Path("/content/meetingbank_artifacts"); OUT_DIR.mkdir(parents=True, exist_ok=True)
ICS_PATH = OUT_DIR / "actions.ics"
print("[OK] Output dir:", OUT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu


[OK] summarizer loaded: sshleifer/distilbart-cnn-12-6


Device set to use cpu


[OK] zero-shot classifier loaded: typeform/distilbert-base-uncased-mnli
[OK] spaCy model loaded
[OK] Output dir: /content/meetingbank_artifacts


In [4]:
# ==== Cell 2: End-to-end pipeline (prints outputs inline) ====
from datasets import load_dataset
import pandas as pd
from icalendar import Calendar, Event
from rouge_score import rouge_scorer
from IPython.display import display, Markdown
import json, re, datetime

# ---------- Speed knobs (tuned for demo) ----------
SAMPLE_SIZE = 1
MAX_CHARS = 9000
CHUNK_CHAR = 1000
MAX_CHUNKS = 2
MAX_SENTENCES_PER_MEETING = 30
SUMMARY_MIN, SUMMARY_MAX = 50, 120
ACTION_THRESHOLD, DECISION_THRESHOLD = 0.60, 0.60

ACTION_HINT = re.compile(
    r"\b(assign|prepare|send|create|update|review|follow\s*up|schedule|organize|draft|share|deliver|finish|"
    r"complete|implement|deploy|fix|investigate|by|before|due|deadline|will|should|must)\b", re.I
)

# ---------- Load MeetingBank (HF mirror) ----------
ds = load_dataset("huuuyeah/meetingbank")   # ['summary','uid','id','transcript']
df = pd.DataFrame(ds["train"][:SAMPLE_SIZE])
print("Using rows:", len(df), " | Columns:", df.columns.tolist())

# ---------- Helpers ----------
import nltk, spacy, dateparser
def split_sents(text: str):
    text = (text or "")[:MAX_CHARS].replace("\n"," ").strip()
    sents = nltk.sent_tokenize(text)
    sents = [s for s in sents if ACTION_HINT.search(s)]
    return sents[:MAX_SENTENCES_PER_MEETING]

def chunk_text(text, chunk_chars=CHUNK_CHAR, max_chunks=MAX_CHUNKS):
    raw = (text or "")[:MAX_CHARS]
    sents = nltk.sent_tokenize(raw)
    chunks, buf = [], ""
    for s in sents:
        if len(buf) + len(s) + 1 <= chunk_chars:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append(buf)
                if len(chunks) >= max_chunks: break
            buf = s
    if buf and len(chunks) < max_chunks:
        chunks.append(buf)
    return chunks

def summarize_long(text):
    parts = []
    for ch in chunk_text(text):
        parts.append(summarizer(ch, max_length=SUMMARY_MAX, min_length=SUMMARY_MIN, do_sample=False)[0]["summary_text"])
    if not parts: return ""
    merged = " ".join(parts)[:2000]
    return summarizer(merged, max_length=SUMMARY_MAX, min_length=SUMMARY_MIN, do_sample=False)[0]["summary_text"]

def classify(sent: str):
    res = zsc(sent, candidate_labels=["action item/task","decision","other"], multi_label=True)
    scores = dict(zip(res["labels"], res["scores"]))
    if scores.get("action item/task",0) >= ACTION_THRESHOLD: return "action", scores
    if scores.get("decision",0)          >= DECISION_THRESHOLD: return "decision", scores
    return "other", scores

date_rgx = re.compile(r"\b(by|before|on|due|next|this|following)\s+([A-Za-z0-9,/\-\s]+)", re.I)
def extract_owner_due(sent: str):
    doc = nlp(sent)
    owner = next((e.text for e in doc.ents if e.label_=="PERSON"), None)
    due = None
    for m in date_rgx.finditer(sent):
        dt = dateparser.parse(m.group(0), settings={"PREFER_DATES_FROM":"future"})
        if dt:
            due = dt.date().isoformat(); break
    return owner, due

def make_ics(actions, path):
    cal = Calendar(); cal.add('prodid','-//Meeting Actions//'); cal.add('version','2.0')
    for a in actions:
        if not a.get("due"): continue
        ev = Event()
        ev.add('summary', a["task"][:160])
        ev.add('description', f"Owner: {a.get('owner') or 'N/A'} | Confidence: {a['confidence_action']:.2f} | Meeting: {a['meeting_id']}")
        ev.add('dtstart', datetime.date.fromisoformat(a["due"]))
        cal.add_component(ev)
    with open(path,'wb') as f: f.write(cal.to_ical())
    return path

# ---------- Run pipeline ----------
summ_rows, actions, decisions = [], [], []

for i, row in df.iterrows():
    mid  = row["uid"] or f"meet_{i}"
    text = (row.get("transcript") or "").strip()
    if not text: continue

    # 1) Summary
    stext = summarize_long(text)
    spath = OUT_DIR / f"{mid}_summary.md"
    with open(spath,"w",encoding="utf-8") as f: f.write(stext)
    summ_rows.append({"meeting_id": mid, "summary_path": str(spath)})

    # PRINT summary inline
    display(Markdown(f"### 📝 Summary for **{mid}**"))
    display(Markdown(stext))

    # 2) Action/Decision extraction
    for s in split_sents(text):
        kind, scores = classify(s)
        if kind == "action":
            owner, due = extract_owner_due(s)
            actions.append({
                "meeting_id": mid, "task": s, "owner": owner, "due": due,
                "confidence_action": float(scores.get("action item/task",0)),
                "evidence_span": s[:280]
            })
        elif kind == "decision":
            decisions.append({
                "meeting_id": mid, "decision": s,
                "confidence_decision": float(scores.get("decision",0)),
                "evidence_span": s[:280]
            })

# Save artifacts
act_path = OUT_DIR/"actions.jsonl"
dec_path = OUT_DIR/"decisions.jsonl"
with open(act_path,"w",encoding="utf-8") as f:
    for a in actions: f.write(json.dumps(a, ensure_ascii=False)+"\n")
with open(dec_path,"w",encoding="utf-8") as f:
    for d in decisions: f.write(json.dumps(d, ensure_ascii=False)+"\n")
ics_file = make_ics(actions, ICS_PATH)

# ---------- PRINT artifacts inline ----------
print(f"\n[OK] Saved artifacts → {act_path} | {dec_path} | {ics_file}")

def read_jsonl(path, limit=10):
    rows = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            rows.append(json.loads(line))
            if i+1 >= limit: break
    return pd.DataFrame(rows)

if os.path.exists(act_path):
    act_df = read_jsonl(act_path, limit=20)
    display(Markdown("### ✅ Extracted Actions (preview)"))
    display(act_df if not act_df.empty else pd.DataFrame(columns=["meeting_id","task","owner","due","confidence_action","evidence_span"]))
else:
    print("No actions.jsonl found.")

if os.path.exists(dec_path):
    dec_df = read_jsonl(dec_path, limit=20)
    display(Markdown("### ✅ Extracted Decisions (preview)"))
    display(dec_df if not dec_df.empty else pd.DataFrame(columns=["meeting_id","decision","confidence_decision","evidence_span"]))
else:
    print("No decisions.jsonl found.")

# ICS preview: count + first few events
try:
    with open(ics_file, "rb") as f:
        cal = Calendar.from_ical(f.read())
    events = [c for c in cal.walk() if c.name == "VEVENT"]
    print(f"\nICS events: {len(events)}")
    for ev in events[:5]:
        print(" -", str(ev.get('summary')), "| dtstart:", ev.get('dtstart').dt)
except Exception as e:
    print("ICS preview skipped:", e)

# ---------- Quick evaluation (ROUGE) ----------
scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
ref_map = { r["uid"]: r["summary"] for _, r in df.iterrows() if isinstance(r.get("summary"), str) and r["summary"].strip() }

rows = []
for s in summ_rows:
    mid = s["meeting_id"]
    ref = (ref_map.get(mid) or "").strip()
    if not ref: continue
    sys_sum = open(s["summary_path"], encoding="utf-8").read()
    sc = scorer.score(ref[:4000], sys_sum[:2000])
    rows.append({"meeting_id": mid,
                 "rouge1_f": sc["rouge1"].fmeasure,
                 "rouge2_f": sc["rouge2"].fmeasure,
                 "rougeL_f": sc["rougeL"].fmeasure})

eval_df = pd.DataFrame(rows)
display(Markdown("### 📊 ROUGE (vs dataset reference)"))
print(eval_df.describe() if not eval_df.empty else "No refs in this sample.")
print("\nArtifacts folder:", OUT_DIR)


Using rows: 1  | Columns: ['summary', 'uid', 'id', 'transcript']


### 📝 Summary for **DenverCityCouncil_05012017_17-0161**

 A pending formal site development plan application may be processed under the provisions of the Denver Zoning Code concerning the small LA parking exemption prior to the adoption of this ordinance . The prior small loft parking exemption . The amendment updates the zoning code reference that changed as a result to the passage of Council Bill 17 .


[OK] Saved artifacts → /content/meetingbank_artifacts/actions.jsonl | /content/meetingbank_artifacts/decisions.jsonl | /content/meetingbank_artifacts/actions.ics


### ✅ Extracted Actions (preview)

...

### ✅ Extracted Decisions (preview)

,meeting_id,decision,confidence_decision,evidence_span



ICS events: 0


### 📊 ROUGE (vs dataset reference)

       rouge1_f  rouge2_f  rougeL_f
count  1.000000  1.000000  1.000000
mean   0.273859  0.083682  0.174274
std         NaN       NaN       NaN
min    0.273859  0.083682  0.174274
25%    0.273859  0.083682  0.174274
50%    0.273859  0.083682  0.174274
75%    0.273859  0.083682  0.174274
max    0.273859  0.083682  0.174274

Artifacts folder: /content/meetingbank_artifacts
